# Capítulo 5 — Evapotranspiração de referência (Hargreaves-Samani × Penman-Monteith)

**Curso:** Agrometeorologia Operacional com Python
**Prof. Dr. Fabrício Correia de Oliveira** — UTFPR, Campus Santa Helena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologia/blob/main/curso/05_evapotranspiracao.ipynb)

> Pré-requisito: Capítulos 1 a 4 (usaremos as funções de radiação e umidade já implementadas).

---


## 5.1 Motivação

A evapotranspiração de referência (`ETo`) é a demanda atmosférica máxima de água — quanto uma
superfície gramada, sem restrição hídrica, perderia por dia. É a variável central de todo o
manejo de irrigação e do balanço hídrico do Capítulo 6.

Existem métodos de complexidade e exigência de dados muito diferentes. Vamos implementar os
dois extremos desse espectro: **Hargreaves-Samani** (só temperatura) e
**Penman-Monteith FAO-56** (padrão-ouro, mais dados, mais precisão).


## 5.2 Fórmulas

**Hargreaves-Samani:**
$$ET_o\ (\text{mm dia}^{-1}) = 0{,}0023 \cdot Q_o^{*} \cdot (T_{max}-T_{min})^{0{,}5} \cdot (T_{med}+17{,}8)$$
em que $Q_o^{*} = Q_o / 2{,}45$ (radiação extraterrestre convertida para mm dia⁻¹ equivalente).

**Penman-Monteith FAO-56:**
$$ET_o = \frac{0{,}408\,s\,(R_n - G) + \gamma\,\dfrac{900}{T_{med}+273}\,U_2\,(\bar{e}_s - e_a)}{s + \gamma\,(1 + 0{,}34\,U_2)}\quad (\text{mm dia}^{-1})$$

com $\gamma = 0{,}063$ kPa °C⁻¹ (constante psicrométrica) e $G \approx 0$ em escala diária.

> Algumas versões da apostila usam `Tmed+275` nesse termo; a `agrometeorologiapy` segue o
> padrão FAO-56 (`Tmed+273`, temperatura em Kelvin). A diferença é pequena (< 0,3% na ETo).

## 5.3 Do papel ao código

Reaproveitamos, sem alterar, as funções da `agrometeorologiapy` já vistas nos Capítulos 2
(radiação) e 4 (umidade). Isso ilustra bem a vantagem de ter uma biblioteca de funções puras
e testadas: evapotranspiração é "apenas" a combinação delas, embutida em `eto_hargreaves_samani`
e `eto_penman_monteith` — dois wrappers que só organizam os parâmetros mais convenientes para
este capítulo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import agrometeorologiapy as amp


def pressao_saturacao_media(Tmax: float, Tmin: float) -> float:
    """ēs — média entre a pressão de saturação em Tmax e em Tmin (kPa) (ver Capítulo 4)."""
    return (amp.es_tetens(Tmax) + amp.es_tetens(Tmin)) / 2


def eto_hargreaves_samani(lat: float, nda: int, Tmax: float, Tmin: float, Tmed: float) -> float:
    declinacao = amp.declinacao_solar(nda)
    Hn = amp.angulo_horario_nascer(lat, declinacao)
    dD2 = amp.fator_correcao_distancia(nda)
    Qo = amp.irradiancia_extraterrestre(lat, declinacao, Hn, dD2)
    return amp.etp_hargreaves_samani(Qo, Tmax, Tmin, Tmed)


def eto_penman_monteith(Tmax: float, Tmin: float, Tmed: float, UR: float,
                          U2: float, Rn: float, G: float = 0.0, gamma: float = 0.063) -> float:
    es_media = pressao_saturacao_media(Tmax, Tmin)
    ea = amp.ea_umidade(es_media, UR)
    Delta = amp.declive_pressao_vapor(Tmed)
    return amp.eto_penman_monteith_fao56(Rn, G, Tmed, U2, es_media, ea, Delta, gamma)

## 5.4 Atividade guiada — reproduzindo os dois exercícios da apostila

**(a) Hargreaves-Samani, ETo mensal para Cascavel-PR (φ = -25°)** em janeiro, abril e julho.
Resultado esperado: ≈ 192,6 / 120,2 / 79,8 mm mês⁻¹.


In [ ]:
lat = -25
meses_hs = pd.DataFrame({
    "mes": ["Janeiro", "Abril", "Julho"],
    "nda": [15, 105, 196],
    "Tmax": [33.0, 28.0, 22.0],
    "Tmin": [21.0, 14.0, 8.0],
    "Tmed": [27.0, 21.0, 15.0],
    "ND":   [31, 30, 31],
})
meses_hs["ETo_diaria"] = meses_hs.apply(
    lambda r: eto_hargreaves_samani(lat, r["nda"], r["Tmax"], r["Tmin"], r["Tmed"]), axis=1
)
meses_hs["ETo_mensal"] = meses_hs["ETo_diaria"] * meses_hs["ND"]
meses_hs.round(2)


**(b) Penman-Monteith, ETo diária para Palotina-PR.** Dados: `Tmax=34,0`, `Tmin=20,0`,
`Tmed=27,0`, `UR=65%`, `U2=2,2 m/s`, `Rn=16,8 MJ m⁻² dia⁻¹`. Resultado esperado ≈ 6,22 mm dia⁻¹.


In [ ]:
ETo_pm = eto_penman_monteith(Tmax=34.0, Tmin=20.0, Tmed=27.0, UR=65, U2=2.2, Rn=16.8)
print(f"ETo (Penman-Monteith) = {ETo_pm:.2f} mm dia-1")


## 5.5 Comparando os dois métodos em dados reais

Hargreaves-Samani só precisa de `Tmax`/`Tmin` — dá para calcular para qualquer estação, mesmo
sem sensores de vento ou umidade. Penman-Monteith é mais completo, mas exige mais variáveis.
Vamos comparar os dois na série de Santa Helena-PR (usando `Rn ≈ Qg × 0,77` como aproximação
simplificada do saldo de radiação líquido — uma simplificação didática; o cálculo completo de
`Rn`, com balanço de ondas longas, é mais elaborado e fica fora do escopo deste capítulo).


In [ ]:
import requests

LAT, LON = -24.86, -54.33
url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M",
    "community": "AG",
    "longitude": LON,
    "latitude": LAT,
    "start": "20230101",
    "end": "20231231",
    "format": "JSON",
}
resposta = requests.get(url, params=params, timeout=60)
resposta.raise_for_status()
propriedades = resposta.json()["properties"]["parameter"]

df_clima = pd.DataFrame(propriedades)
df_clima.index = pd.to_datetime(df_clima.index, format="%Y%m%d")
df_clima.index.name = "data"
df_clima = df_clima.replace(-999, np.nan)
df_clima["nda"] = df_clima.index.dayofyear

df_clima["ETo_HS"] = df_clima.apply(
    lambda r: eto_hargreaves_samani(LAT, r["nda"], r["T2M_MAX"], r["T2M_MIN"], r["T2M"]), axis=1
)
df_clima["Rn_aprox"] = df_clima["ALLSKY_SFC_SW_DWN"] * 0.77  # simplificação didática
df_clima["ETo_PM"] = df_clima.apply(
    lambda r: eto_penman_monteith(r["T2M_MAX"], r["T2M_MIN"], r["T2M"], r["RH2M"], r["WS2M"], r["Rn_aprox"]),
    axis=1
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_clima.index, df_clima["ETo_HS"], label="Hargreaves-Samani", alpha=0.8)
ax.plot(df_clima.index, df_clima["ETo_PM"], label="Penman-Monteith (Rn aproximado)", alpha=0.8)
ax.set_ylabel("ETo (mm dia-1)")
ax.set_title("Santa Helena-PR — comparação de métodos de ETo (2023)")
ax.legend()
plt.tight_layout()
plt.show()

print("Correlação entre os dois métodos:", df_clima[["ETo_HS", "ETo_PM"]].corr().iloc[0, 1].round(3))


## 5.6 Desafio — análise de sensibilidade

1. Recalcule a ETo de Palotina-PR (item 5.4b) variando `UR` em ±10% (58,5% e 71,5%),
   mantendo tudo o mais constante. Quanto a ETo muda?
2. Repita variando `U2` em ±10% em vez de `UR`. Qual variável tem maior impacto na ETo —
   umidade relativa ou vento? Monte uma tabela comparando os quatro cenários.


In [ ]:
# Espaço para o desafio — escreva seu código aqui


## 5.7 Checkpoint

Antes de seguir para o **Capítulo 6 — Balanço hídrico climatológico**, você deve ter:

- [ ] reproduzido os dois exercícios da apostila (HS mensal e PM diário) com os mesmos valores;
- [ ] funções `eto_hargreaves_samani` e `eto_penman_monteith` testadas;
- [ ] uma série de ETo diária de um ano inteiro para uma estação real — ela vira a entrada
      (`ETP`) do balanço hídrico do próximo capítulo.
